# 4. Rebase

Merging joins two branches with a merge commit. **Rebasing** moves your
branch so it starts from the newest `main`, as if you had started your work
later. The history stays a straight line.

Before:

```mermaid
gitGraph
    commit id: "volume"
    branch tpc
    commit id: "tonnes per cm"
    checkout main
    commit id: "units"
```

After `git rebase main`:

```mermaid
gitGraph
    commit id: "volume"
    commit id: "units"
    branch tpc
    commit id: "tonnes per cm (copy)"
```

In [ ]:
!rm -rf ~/git-course/04-rebase
!mkdir -p ~/git-course/04-rebase
%cd ~/git-course/04-rebase

## Starting point

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

A branch `tpc` with one commit. (There is a mistake in it: it divides by
1000 instead of 100000. We fix that later.)

In [ ]:
!git switch -c tpc

In [ ]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 1000.0

In [ ]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

Meanwhile, `main` gets a new commit:

In [ ]:
!git switch main

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git add hydrostatics.py
!git commit -m "Say which units the module uses"

In [ ]:
!git log --oneline --graph --all

## Rebasing

On `tpc`, `git rebase main` replays our commit on top of the newest `main`:

In [ ]:
!git switch tpc
!git rebase main

In [ ]:
!git log --oneline --graph --all

A straight line. Look at the commit's id (the short hash): it changed.
Rebase does not move commits, it makes **copies** of them on the new
starting point.

## Tidying up: fixing a commit

Now we fix the mistake (100000, not 1000):

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Instead of a separate "fix typo" commit, we mark it as a fix for the last
commit with `--fixup`:

In [ ]:
!git add hydrostatics.py
!git commit --fixup HEAD

In [ ]:
!git log --oneline

Then an **interactive rebase** with `--autosquash` folds the fix into the
commit it belongs to.

Normally `git rebase -i` opens an editor so you can review the plan. A
notebook cannot open an editor, so `GIT_SEQUENCE_EDITOR=true` says "accept
the plan as it is".

In [ ]:
!GIT_SEQUENCE_EDITOR=true git rebase -i --autosquash main

In [ ]:
!git log --oneline

One clean commit, as if we had never made the mistake.

Now `main` can take the branch with a fast-forward — no merge commit:

In [ ]:
!git switch main
!git merge tpc

In [ ]:
!git log --oneline --graph --all

## When a rebase has a conflict

It works like a merge conflict: fix the file, `git add` it, then
`git rebase --continue`. Or give up with `git rebase --abort`.

## The golden rule

**Never rebase commits other people already have.** Rebase makes new
copies of commits; if a colleague has the old ones, your histories no
longer match. Rebase only your own work that you have not pushed yet.

## Rebase or merge?

| | Merge | Rebase |
| - | - | - |
| History | keeps the branch shape | one straight line |
| Commits | unchanged | replaced by copies |
| Safe on shared branches | yes | no |

## Exercises

1. Make two commits on a branch, move `main` forward, and rebase the
   branch.
2. Make a commit, then a `--fixup` commit, and fold them together with
   `git rebase -i --autosquash`.

Next: [5. Remotes and origin](05-remotes.ipynb)